# Задание 8.2 — Fine-tuning T5-Small (Zerocoder, модуль 8)

Все 11 шагов домашнего задания в одном ноутбуке. Работает на **двух датасетах**:

* `DATASET = "xsum"` — эталон задания: новости BBC → саммари в одно предложение;
* `DATASET = "normassist"` — собственный датасет из **задания 8.1** (нормативный контекст → краткий ответ со ссылками `[n]`).

> **Среда выполнения:** Runtime → Change runtime type → **T4 GPU**.
> Без GPU ноутбук тоже отработает, но медленнее и с `fp16=False`.

Токен Hugging Face и ключ wandb **не нужны**: `push_to_hub=False`, `report_to=[]`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Шаг 1. Установка библиотек и инструментов

In [11]:
!pip -q install "transformers>=4.44" "datasets>=3.0" "evaluate>=0.4" rouge-score nltk sentencepiece accelerate

import sys, os, time, json
import torch, transformers, datasets, evaluate
import numpy as np

for k, v in {"python": sys.version.split()[0], "torch": torch.__version__,
             "transformers": transformers.__version__, "datasets": datasets.__version__,
             "evaluate": evaluate.__version__}.items():
    print(f"{k:<15} {v}")

HAS_GPU = torch.cuda.is_available()
print("\nУстройство:", torch.cuda.get_device_name(0) if HAS_GPU else f"CPU ({os.cpu_count()} ядер)")
if HAS_GPU:
    !nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

python          3.12.13
torch           2.11.0+cu128
transformers    5.13.1
datasets        4.0.0
evaluate        0.4.6

Устройство: Tesla T4
name, memory.total [MiB], memory.used [MiB]
Tesla T4, 15360 MiB, 207 MiB


## Шаг 2. Определяем модель T5-Small и датасет

`DATASET = "xsum"` — как у эксперта. Поставь `"normassist"`, чтобы обучаться на собственных
данных из задания 8.1 (файлы `train.jsonl` / `validation.jsonl` из папки
`Perr8.2/data/normassist_seq2seq/` нужно загрузить в Colab через **Files → Upload**).

In [12]:
model_checkpoint = "t5-small"
DATASET = "xsum"          # "xsum" или "normassist"
prefix = "summarize: "

# Размеры выборки: урок сам предлагает уменьшать датасет под лимиты Colab.
N_TRAIN, N_VAL = 5000, 1000
EPOCHS, BATCH_SIZE, LR = 1, 16, 2e-5
MAX_INPUT, MAX_TARGET = 1024, 128

print(f"Модель:  {model_checkpoint}")
print(f"Датасет: {DATASET}")
print(f"Префикс задачи для T5: {prefix!r}")

Модель:  t5-small
Датасет: xsum
Префикс задачи для T5: 'summarize: '


## Шаг 3. Загружаем датасет и метрику ROUGE, смотрим структуру

In [13]:
from datasets import load_dataset

if DATASET == "xsum":
    raw_datasets = load_dataset("EdinburghNLP/xsum")
else:
    raw_datasets = load_dataset("json", data_files={"train": "train.jsonl",
                                                    "validation": "validation.jsonl"})
metric = evaluate.load("rouge")

print(raw_datasets)
print("\nПример document:", raw_datasets["train"][0]["document"][:400])
print("\nПример summary :", raw_datasets["train"][0]["summary"][:300])

DatasetDict({
    train: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 204045
    })
    validation: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 11332
    })
    test: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 11334
    })
})

Пример document: The full cost of damage in Newton Stewart, one of the areas worst affected, is still being assessed.
Repair work is ongoing in Hawick and many roads in Peeblesshire remain badly affected by standing water.
Trains on the west coast mainline face disruption due to damage at the Lamington Viaduct.
Many businesses and householders were affected by flooding in Newton Stewart after the River Cree overfl

Пример summary : Clean-up operations are continuing across the Scottish Borders and Dumfries and Galloway after flooding caused by Storm Frank.


In [14]:
# Уменьшаем размер датасета под ресурсы Colab
raw_datasets["train"] = raw_datasets["train"].shuffle(seed=42).select(range(min(N_TRAIN, len(raw_datasets["train"]))))
raw_datasets["validation"] = raw_datasets["validation"].shuffle(seed=42).select(range(min(N_VAL, len(raw_datasets["validation"]))))
print({k: len(v) for k, v in raw_datasets.items()})

{'train': 5000, 'validation': 1000, 'test': 11334}


## Шаг 4. Токенизация данных

Главное правило урока: **токенизатор берём от той же модели**. Ячейка ниже это же и проверяет
на своих данных — считает долю `<unk>` (символов, которых нет в словаре модели).

In [15]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
print("Словарь:", tokenizer.vocab_size, "токенов")
print(tokenizer("Hello, this one sentence!"))

def preprocess_function(examples):
    inputs = [prefix + doc for doc in examples["document"]]
    model_inputs = tokenizer(inputs, max_length=MAX_INPUT, truncation=True)
    labels = tokenizer(text_target=examples["summary"], max_length=MAX_TARGET, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

demo = preprocess_function(raw_datasets["train"][:2])
print("\ninput_ids[0][:24] =", demo["input_ids"][0][:24])
print("labels[0][:24]    =", demo["labels"][0][:24])

ids = [i for x in demo["input_ids"] for i in x]
print(f"доля <unk>: {sum(i == tokenizer.unk_token_id for i in ids) / len(ids):.2%}")

tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)
tokenized_datasets

Словарь: 32100 токенов
{'input_ids': [8774, 6, 48, 80, 7142, 55, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}

input_ids[0][:24] = [21603, 10, 86, 10256, 6, 6098, 7, 33, 1966, 21, 3135, 11, 12162, 53, 2061, 5, 299, 16, 2789, 6, 1363, 411, 7, 12940]
labels[0][:24]    = [282, 3, 26767, 3080, 411, 7, 12940, 2162, 66, 1566, 538, 2061, 56, 582, 3, 9, 6615, 2720, 7, 6, 8, 22982, 3141, 3256]
доля <unk>: 0.00%


DatasetDict({
    train: Dataset({
        features: ['document', 'summary', 'id', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 5000
    })
    validation: Dataset({
        features: ['document', 'summary', 'id', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['document', 'summary', 'id', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 11334
    })
})

## Шаг 5. Инициализируем модель T5-Small

In [16]:
from transformers import (AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
                          Seq2SeqTrainingArguments, Seq2SeqTrainer)

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
total = sum(p.numel() for p in model.parameters())
print(type(model).__name__)
print(f"Параметров: {total:,} — обучаются все (полный fine-tuning, без LoRA)")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

T5ForConditionalGeneration
Параметров: 60,506,624 — обучаются все (полный fine-tuning, без LoRA)


## Шаг 6. Задаём гиперпараметры обучения

In [17]:
model_name = model_checkpoint.split("/")[-1]
args = Seq2SeqTrainingArguments(
    output_dir=f"{model_name}-finetuned-{DATASET}",
    eval_strategy="epoch",
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=EPOCHS,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET,
    fp16=HAS_GPU,        # fp16 только на GPU
    push_to_hub=False,   # ничего не публикуем -> токен HF не нужен
    report_to=[],        # без wandb -> ключ wandb не нужен
)
args

Seq2SeqTrainingArguments(output_dir='t5-small-finetuned-xsum', per_device_train_batch_size=16, num_train_epochs=1, max_steps=-1, learning_rate=2e-05, lr_scheduler_type=<SchedulerType.LINEAR: 'linear'>, lr_scheduler_kwargs=None, warmup_steps=0, optim=<OptimizerNames.ADAMW_TORCH_FUSED: 'adamw_torch_fused'>, optim_args=None, weight_decay=0.01, adam_beta1=0.9, adam_beta2=0.999, adam_epsilon=1e-08, optim_target_modules=None, gradient_accumulation_steps=1, average_tokens_across_devices=True, max_grad_norm=1.0, label_smoothing_factor=0.0, bf16=False, fp16=True, bf16_full_eval=False, fp16_full_eval=False, tf32=None, gradient_checkpointing=False, gradient_checkpointing_kwargs=None, torch_compile=False, torch_compile_backend=None, torch_compile_mode=None, use_liger_kernel=False, liger_kernel_config=None, use_cache=False, neftune_noise_alpha=None, torch_empty_cache_steps=None, auto_find_batch_size=False, logging_strategy=<IntervalStrategy.STEPS: 'steps'>, logging_steps=500, logging_first_step=Fal

In [19]:
tokenized_datasets = raw_datasets.map(
    preprocess_function, batched=True,
    remove_columns=raw_datasets["train"].column_names)
print("колонки после токенизации:", tokenized_datasets["train"].column_names)


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/11334 [00:00<?, ? examples/s]

колонки после токенизации: ['input_ids', 'attention_mask', 'labels']


## Шаг 7. Создаём упаковщик данных

In [20]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
batch = data_collator([tokenized_datasets["train"][i] for i in range(4)])
{k: tuple(v.shape) for k, v in batch.items()}

{'input_ids': (4, 956),
 'attention_mask': (4, 956),
 'labels': (4, 31),
 'decoder_input_ids': (4, 31)}

## Шаг 8. Функция вычисления метрик ROUGE (+ своя метрика длины)

In [21]:
import nltk
for pkg in ("punkt_tab", "punkt"):
    try: nltk.download(pkg, quiet=True)
    except Exception: pass

def split_sentences(text):
    try:
        return nltk.sent_tokenize(text)
    except Exception:
        return [s for s in text.replace(". ", ".\n").split("\n") if s]

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds = ["\n".join(split_sentences(p.strip())) for p in decoded_preds]
    decoded_labels = ["\n".join(split_sentences(l.strip())) for l in decoded_labels]
    result = metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    result = {k: v * 100 for k, v in result.items()}
    result["gen_len"] = float(np.mean([np.count_nonzero(p != tokenizer.pad_token_id) for p in predictions]))
    return {k: round(float(v), 4) for k, v in result.items()}

## Шаг 9. Передаём всё в Seq2SeqTrainer

In [22]:
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
print(f"шагов на эпоху: ~{len(tokenized_datasets['train']) // BATCH_SIZE}")

шагов на эпоху: ~312


## Шаг 10. Запускаем обучение

Сначала — замер **до** обучения. Без базовой линии нельзя сказать, что именно дало дообучение.

In [23]:
baseline = trainer.evaluate(metric_key_prefix="base")
print({k: v for k, v in baseline.items() if "rouge" in k or "loss" in k or "gen_len" in k})

Training Loss,Validation Loss,Epoch,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
No log,3.849231,0,19.120600,2.785800,13.490000,15.466700,53.660000


{'base_loss': 3.849231004714966, 'base_rouge1': 19.1206, 'base_rouge2': 2.7858, 'base_rougeL': 13.49, 'base_rougeLsum': 15.4667, 'base_gen_len': 53.66}


In [24]:
t0 = time.time()
train_result = trainer.train()
train_time = time.time() - t0
print(f"\nОбучение заняло {train_time/60:.1f} мин, training loss = {train_result.training_loss:.4f}")

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,No log,2.820921,21.484000,3.899100,16.166000,16.389700,29.268000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Обучение заняло 5.8 мин, training loss = 3.1950


## Шаг 11. Отслеживаем метрики: loss, ROUGE, длина, время, GPU

In [25]:
final = trainer.evaluate()

print(f"{'эпоха':>6} {'val loss':>10} {'rouge1':>8} {'rouge2':>8} {'rougeL':>8} {'gen_len':>8}")
for h in trainer.state.log_history:
    if "eval_loss" in h:
        print(f"{h['epoch']:>6.2f} {h['eval_loss']:>10.4f} {h.get('eval_rouge1',0):>8.2f} "
              f"{h.get('eval_rouge2',0):>8.2f} {h.get('eval_rougeL',0):>8.2f} {h.get('eval_gen_len',0):>8.1f}")

print("\nДО обучения -> ПОСЛЕ обучения:")
for m in ("loss", "rouge1", "rouge2", "rougeL", "gen_len"):
    b, f = baseline.get(f"base_{m}"), final.get(f"eval_{m}")
    if b is not None and f is not None:
        print(f"  {m:<10}{b:>9.3f} -> {f:>9.3f}   ({f-b:+.3f})")

print(f"\nВремя обучения: {train_time/60:.1f} мин")
if HAS_GPU:
    print(f"Пик VRAM: {torch.cuda.max_memory_allocated()/1e9:.2f} ГБ")
    !nvidia-smi --query-gpu=name,memory.used,utilization.gpu --format=csv

Training Loss,Validation Loss,Epoch,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
No log,2.820921,1,21.484000,3.899100,16.166000,16.389700,29.268000


 эпоха   val loss   rouge1   rouge2   rougeL  gen_len
  1.00     2.8209    21.48     3.90    16.17     29.3
  1.00     2.8209    21.48     3.90    16.17     29.3

ДО обучения -> ПОСЛЕ обучения:
  loss          3.849 ->     2.821   (-1.028)
  rouge1       19.121 ->    21.484   (+2.363)
  rouge2        2.786 ->     3.899   (+1.113)
  rougeL       13.490 ->    16.166   (+2.676)
  gen_len      53.660 ->    29.268   (-24.392)

Время обучения: 5.8 мин
Пик VRAM: 11.78 ГБ
name, memory.used [MiB], utilization.gpu [%]
Tesla T4, 14707 MiB, 0 %


In [27]:
# Живые примеры генерации — глазами, а не только по метрике
model.eval()
for i in range(3):
    row = raw_datasets["validation"][i]
    enc = tokenizer(prefix + row["document"], max_length=MAX_INPUT,
                    truncation=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_length=MAX_TARGET, num_beams=1)
    print(f"--- пример {i+1} ---")
    print("эталон:        ", row["summary"][:200])
    print("модель выдала: ", tokenizer.decode(out[0], skip_special_tokens=True)[:200], "\n")

--- пример 1 ---
эталон:         Three family members have been jailed for forcing a man to do heavy labour for tiny amounts in Cardiff.
модель выдала:  Patrick Joseph Connors, 59, his son Patrick Dean Connors, 39, and nephew William Connors, 36, were convicted of 'exploiting and controlling' their vulnerable victim in a "callous manner over a prolong 

--- пример 2 ---
эталон:         Championship leaders Hibernian twice came from behind to salvage a draw at home to Dumbarton.
модель выдала:  Hibs have won their last 10 games in the premier league this season. the visitors have won their last three games in the premier league this season. 

--- пример 3 ---
эталон:         A girl stolen as a newborn from a hospital in Jacksonville, Florida, has been found alive in South Carolina after more than 18 years, police say.
модель выдала:  Gloria Williams, 51, was abducted in 1998 by a woman posing as a health care worker at the University Medical Center in Walterboro, South Carolina. 



In [28]:
# Сохраняем все метрики рядом с ноутбуком
json.dump({"dataset": DATASET, "train_time_sec": round(train_time, 1),
           "train_loss": train_result.training_loss,
           "baseline": baseline, "final": final,
           "log_history": trainer.state.log_history},
          open(f"metrics_{DATASET}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("сохранено:", f"metrics_{DATASET}.json")

сохранено: metrics_xsum.json
